## 6. N-BEATS Baseline

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
    !pip install -q pytorch_forecasting
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

IN_COLAB=False  REPO_ROOT=/Users/ucabtt4/Projects/Dissertation/dissertation


In [6]:
import json

import numpy as np

from src.dataloader import load_cgm_stats, load_tensors, load_participant_ids
from src import metrics
from src import train
from src.models.nn_baseline import NBEATSPredictor, HORIZONS

MODEL_NAME = 'NBEATS'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'nbeats'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'nbeats_stability_summary.json'

In [7]:
stats = load_cgm_stats()
train_loader = load_tensors('train')  # batch_size=256 default (Training Procedure)
val_loader = load_tensors('val')
test_loader = load_tensors('test')

test_participant_ids = load_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print(f'number of train batches: ', len(train_loader), '\nnumber of valuation batches: ', len(val_loader),
      '\nnumber of test batches: ', len(test_loader))
print('number of test participants:', len(PARTICIPANT_IDS))
print('Corresponding participants ID: \n', PARTICIPANT_IDS)

sample_cgm, sample_target = next(iter(train_loader))
print('\ninput batch shape (cgm):', tuple(sample_cgm.shape))
print('target batch shape:', tuple(sample_target.shape))

number of train batches:  327 
number of valuation batches:  66 
number of test batches:  66
number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

input batch shape (cgm): (256, 24)
target batch shape: (256, 5)


In [5]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []  # one entry per seed, in memory -- also written to disk individually below

for seed in SEEDS:
    print(f"Test results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)
    model = NBEATSPredictor(context_length=sample_cgm.shape[1])

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / f'nbeats_seed{seed}',
        verbose=False,
        progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader)
    y_pred_test, y_true_test = train.predict(model, test_loader)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])

    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'nbeats_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = test_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")


----------------seed=7----------------
device: cpu  model: NBEATSPredictor  params: 75,312


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

seed=  7  stopped at epoch  16/50  test RMSE @ [15, 30, 60, 90, 120] min = [11.83, 18.98, 27.34, 32.48, 35.39]

----------------seed=14----------------
device: cpu  model: NBEATSPredictor  params: 75,312


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 14  stopped at epoch  35/50  test RMSE @ [15, 30, 60, 90, 120] min = [11.49, 18.58, 27.22, 32.32, 35.2]

----------------seed=21----------------
device: cpu  model: NBEATSPredictor  params: 75,312


...

seed= 56  stopped at epoch  21/50  test RMSE @ [15, 30, 60, 90, 120] min = [11.78, 18.63, 27.08, 32.16, 35.05]

----------------seed=63----------------
device: cpu  model: NBEATSPredictor  params: 75,312


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 63  stopped at epoch  43/50  test RMSE @ [15, 30, 60, 90, 120] min = [11.45, 18.46, 27.07, 32.27, 35.23]

----------------seed=70----------------
device: cpu  model: NBEATSPredictor  params: 75,312


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 70  stopped at epoch  45/50  test RMSE @ [15, 30, 60, 90, 120] min = [11.28, 18.38, 26.91, 32.14, 35.12]


In [ ]:
def _extract(participant_result, metric):
    """Extract a metric from one participant's result."""
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

summary = {}

for h in HORIZONS:
    summary[h] = {}
    for m in METRICS:
        if m == 'zone_a':
            seed_values = [
                r['test_results'][h]['ceg']['A']
            for r in all_results
            ]
        elif m == 'zone_ab':
            seed_values = [
                r['test_results'][h]['ceg']['A']
                + r['test_results'][h]['ceg']['B']
                for r in all_results
            ]
        else:
            seed_values = [
                r['test_results'][h][m]
                for r in all_results
            ]

        mean = float(np.mean(seed_values))
        sd1 = float(np.std(seed_values, ddof=1))

        participant_means = []
        for pid in PARTICIPANT_IDS:
            values_across_seeds = [
                _extract(r['test_results_per_participant'][pid][h], m)
                for r in all_results
            ]
            participant_means.append(np.mean(values_across_seeds))
        sd2 = float(np.std(participant_means, ddof=1))
        summary[h][m] = {
            'mean': mean,
            'sd1_across_seeds': sd1,
            'sd2_across_participants': sd2,
        }

print(
    f'=== {MODEL_NAME} test set: '
    f'overall mean +/- SD_seed (SD_participant) ==='
)

for h in HORIZONS:
    s = summary[h]
    print(
        f"{h:>3}-min:  "
        f"RMSE = {s['rmse']['mean']:.2f} +/- "
        f"{s['rmse']['sd1_across_seeds']:.2f} "
        f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
        f"MAE = {s['mae']['mean']:.2f} +/- "
        f"{s['mae']['sd1_across_seeds']:.2f} "
        f"({s['mae']['sd2_across_participants']:.2f})   "
        f"gRMSE = {s['grmse']['mean']:.2f} +/- "
        f"{s['grmse']['sd1_across_seeds']:.2f} "
        f"({s['grmse']['sd2_across_participants']:.2f})   "
        f"Zone A = {s['zone_a']['mean']:.2f} +/- "
        f"{s['zone_a']['sd1_across_seeds']:.2f} "
        f"({s['zone_a']['sd2_across_participants']:.2f})%   "
        f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- "
        f"{s['zone_ab']['sd1_across_seeds']:.2f} "
        f"({s['zone_ab']['sd2_across_participants']:.2f})%"
    )


In [ ]:
with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}/nbeats_seed*_results.json')
print(f'saved aggregate summary to {STABILITY_SUMMARY_PATH}')